In [ ]:
!pip -q install transformers pandas
!git clone -q https://github.com/aetting/lm-diagnostics.git

In [ ]:
import os, re, torch, numpy as np, pandas as pd
from transformers import BertTokenizer, BertForMaskedLM

REPO = "/content/lm-diagnostics"
SIMP_PATH = f"{REPO}/datasets/NEG-136/NEG-136-SIMP.tsv"
NAT_PATH  = f"{REPO}/datasets/NEG-136/NEG-136-NAT.tsv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

simp_raw = pd.read_csv(SIMP_PATH, sep="\t")
nat_raw  = pd.read_csv(NAT_PATH, sep="\t")

print("SIMP:", simp_raw.shape)  # (18, 5)
print("NAT :", nat_raw.shape)   # (16, 6)
display(simp_raw.head())
display(nat_raw.head())

device: cuda
SIMP: (18, 5)
NAT : (16, 6)


,item,context_aff,context_neg,target_aff,target_neg
0,0,A trout is (a|an),A trout is not (a|an),fish,tool
1,1,A salmon is (a|an),A salmon is not (a|an),fish,flower
2,2,An ant is (a|an),An ant is not (a|an),insect,vegetable
3,3,A bee is (a|an),A bee is not (a|an),insect,building
4,4,A robin is (a|an),A robin is not (a|an),bird,tree


,item,context_aff,context_neg,target_aff,target_neg,licensing
0,0,"With proper equipment, scuba-diving is very","With proper equipment, scuba-diving isn't very",safe,dangerous,Y
1,1,Traveling in Baghdad is very,Traveling in Baghdad isn't very,dangerous,safe,Y
2,2,"In moderation, drinking red wine is","In moderation, drinking red wine isn't",good,bad,Y
3,3,Drinking tap water in developing countries is ...,Drinking tap water in developing countries isn...,dangerous,safe,Y
4,4,Secondhand furniture is very,Secondhand furniture isn't very,cheap,expensive,Y


In [ ]:
def process_negsimp(path):
    rows = []
    df = pd.read_csv(path, sep="\t")

    for _, r in df.iterrows():
        affsent = re.sub(r" \(.+\)", "", r["context_aff"])
        negsent = re.sub(r" \(.+\)", "", r["context_neg"])
        afftgt = r["target_aff"].strip()
        negtgt = r["target_neg"].strip()

        for sent, tgt, cond in [
            (affsent, afftgt, "TA"),
            (negsent, afftgt, "FN"),
            (affsent, negtgt, "FA"),
            (negsent, negtgt, "TN"),
        ]:
            det = "an" if re.match(r"[aeiou]", tgt) else "a"
            context = f"{sent} {det} [MASK]"
            expected = afftgt if cond in ("TA", "FA") else negtgt
            rows.append({
                "item": str(r["item"]),
                "cond": cond,
                "context": context,
                "target": tgt,
                "expected": expected,
            })
    return pd.DataFrame(rows)


def process_negnat(path):
    rows = []
    df = pd.read_csv(path, sep="\t")

    for _, r in df.iterrows():
        affsent = r["context_aff"].strip()
        negsent = r["context_neg"].strip()
        afftgt = r["target_aff"].strip()
        negtgt = r["target_neg"].strip()

        for sent, tgt, cond in [
            (affsent, afftgt, "TA"),
            (negsent, afftgt, "FN"),
            (affsent, negtgt, "FA"),
            (negsent, negtgt, "TN"),
        ]:
            context = f"{sent} [MASK]"   # IMPORTANT: no a/an here
            expected = afftgt if cond in ("TA", "FA") else negtgt
            rows.append({
                "item": str(r["item"]),
                "cond": cond,
                "context": context,
                "target": tgt,
                "expected": expected,
                "licensing": r["licensing"],  # Y = natural; N = less natural
            })
    return pd.DataFrame(rows)

simp = process_negsimp(SIMP_PATH)
nat = process_negnat(NAT_PATH)

assert len(simp) == 18 * 4
assert len(nat) == 16 * 4
assert set(simp.cond) == {"TA","FN","FA","TN"}
assert set(nat.cond) == {"TA","FN","FA","TN"}

display(simp.head(8))
display(nat.head(8))

,item,cond,context,target,expected
0,0,TA,A trout is a [MASK],fish,fish
1,0,FN,A trout is not a [MASK],fish,tool
2,0,FA,A trout is a [MASK],tool,fish
3,0,TN,A trout is not a [MASK],tool,tool
4,1,TA,A salmon is a [MASK],fish,fish
5,1,FN,A salmon is not a [MASK],fish,flower
6,1,FA,A salmon is a [MASK],flower,fish
7,1,TN,A salmon is not a [MASK],flower,flower


,item,cond,context,target,expected,licensing
0,0,TA,"With proper equipment, scuba-diving is very [M...",safe,safe,Y
1,0,FN,"With proper equipment, scuba-diving isn't very...",safe,dangerous,Y
2,0,FA,"With proper equipment, scuba-diving is very [M...",dangerous,safe,Y
3,0,TN,"With proper equipment, scuba-diving isn't very...",dangerous,dangerous,Y
4,1,TA,Traveling in Baghdad is very [MASK],dangerous,dangerous,Y
5,1,FN,Traveling in Baghdad isn't very [MASK],dangerous,safe,Y
6,1,FA,Traveling in Baghdad is very [MASK],safe,dangerous,Y
7,1,TN,Traveling in Baghdad isn't very [MASK],safe,safe,Y


In [ ]:
def load_bert(name):
    tok = BertTokenizer.from_pretrained(name)
    model = BertForMaskedLM.from_pretrained(name).to(device)
    model.eval()
    return tok, model

tok_base, bert_base = load_bert("bert-base-uncased")
tok_large, bert_large = load_bert("bert-large-uncased")
print("Models loaded.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-large-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded.


In [ ]:
def prep_input(sentence, tokenizer):
    text = ["[CLS]"] + sentence.strip().split()
    if text[-1] != ".":
        text.append(".")
    text.append("[SEP]")

    tokenized = tokenizer.tokenize(" ".join(text))
    mask_positions = [i for i, t in enumerate(tokenized) if t == "[MASK]"]
    assert len(mask_positions) == 1, (sentence, tokenized)

    ids = tokenizer.convert_tokens_to_ids(tokenized)
    input_ids = torch.tensor([ids], device=device)
    return input_ids, mask_positions[0], tokenized


@torch.inference_mode()
def score_dataframe(df, model, tokenizer, topk=5):
    out = df.copy()
    target_probs, top_tokens, top_probs = [], [], []

    for _, row in out.iterrows():
        input_ids, mi, toks = prep_input(row["context"], tokenizer)
        logits = model(input_ids=input_ids).logits[0, mi]
        probs = torch.softmax(logits, dim=-1)

        # NEG-136 targets are intended to be single BERT vocabulary tokens.
        target = row["target"]
        pieces = tokenizer.tokenize(target)
        assert len(pieces) == 1 and pieces[0] == target, (target, pieces)
        target_id = tokenizer.convert_tokens_to_ids(target)
        target_probs.append(probs[target_id].item())

        vals, inds = torch.topk(probs, k=topk)
        top_tokens.append(tokenizer.convert_ids_to_tokens(inds.tolist()))
        top_probs.append(vals.detach().cpu().tolist())

    out["target_prob"] = target_probs
    out["top_tokens"] = top_tokens
    out["top_probs"] = top_probs
    return out

In [ ]:
# Sanity check: this should look like "[CLS] a robin is a [MASK] . [SEP]"
example = simp.query("item == '4' and cond == 'TA'").iloc[0]
_, mi, toks = prep_input(example.context, tok_large)
print(example.context)
print(toks)
print("mask index:", mi)

A robin is a [MASK]
['[CLS]', 'a', 'robin', 'is', 'a', '[MASK]', '.', '[SEP]']
mask index: 5


In [ ]:
simp_base  = score_dataframe(simp, bert_base,  tok_base,  topk=5)
simp_large = score_dataframe(simp, bert_large, tok_large, topk=5)
nat_base   = score_dataframe(nat,  bert_base,  tok_base,  topk=5)
nat_large  = score_dataframe(nat,  bert_large, tok_large, topk=5)

print("Scoring complete.")

Scoring complete.


In [ ]:
def simp_accuracy(scored, k):
    ta = scored[scored.cond == "TA"].copy()
    hits = [
        int(row.expected in row.top_tokens[:k])
        for _, row in ta.iterrows()
    ]
    return 100 * np.mean(hits), sum(hits), len(hits)

table11 = pd.DataFrame([
    ["BERT Base",  *simp_accuracy(simp_base, 1), *simp_accuracy(simp_base, 5)],
    ["BERT Large", *simp_accuracy(simp_large, 1), *simp_accuracy(simp_large, 5)],
], columns=["Model", "k1_pct", "k1_correct", "k1_n", "k5_pct", "k5_correct", "k5_n"])

display(table11)

print("Paper target:")
print("BERT Base : k=1 38.9%, k=5 100%")
print("BERT Large: k=1 44.4%, k=5 100%")

,Model,k1_pct,k1_correct,k1_n,k5_pct,k5_correct,k5_n
0,BERT Base,38.888889,7,18,100.0,18,18
1,BERT Large,44.444444,8,18,100.0,18,18


Paper target:
BERT Base : k=1 38.9%, k=5 100%
BERT Large: k=1 44.4%, k=5 100%


In [ ]:
def wide_probs(scored):
    return (
        scored.pivot(index="item", columns="cond", values="target_prob")
              .reset_index()
    )

def neg_sensitivity(scored, threshold=None):
    w = wide_probs(scored)

    aff_diff = w["TA"] - w["FA"]
    neg_diff = w["TN"] - w["FN"]

    if threshold is None:
        aff_hit = aff_diff > 0
        neg_hit = neg_diff > 0
    else:
        aff_hit = aff_diff > threshold
        neg_hit = neg_diff > threshold

    return {
        "Affirmative": 100 * aff_hit.mean(),
        "Negative": 100 * neg_hit.mean(),
        "Aff_correct": int(aff_hit.sum()),
        "Neg_correct": int(neg_hit.sum()),
        "n": len(w),
    }

table12 = pd.DataFrame([
    {"Model": "BERT Base",  **neg_sensitivity(simp_base)},
    {"Model": "BERT Large", **neg_sensitivity(simp_large)},
])
display(table12)

print("Paper target: both models = 100% affirmative, 0% negative")

# The paper also discusses a 0.01 probability-difference threshold.
table12_threshold = pd.DataFrame([
    {"Model": "BERT Base",  **neg_sensitivity(simp_base, threshold=.01)},
    {"Model": "BERT Large", **neg_sensitivity(simp_large, threshold=.01)},
])
display(table12_threshold)

,Model,Affirmative,Negative,Aff_correct,Neg_correct,n
0,BERT Base,100.0,0.0,18,0,18
1,BERT Large,100.0,0.0,18,0,18


Paper target: both models = 100% affirmative, 0% negative


,Model,Affirmative,Negative,Aff_correct,Neg_correct,n
0,BERT Base,94.444444,0.0,17,0,18
1,BERT Large,100.000000,0.0,18,0,18


In [ ]:
def nat_table14(scored):
    meta = scored[["item", "licensing"]].drop_duplicates("item")
    w = wide_probs(scored).merge(meta, on="item", how="left")

    result = {}
    for lic, label in [("Y", "NT"), ("N", "LN")]:
        d = w[w.licensing == lic]
        result[f"Aff. {label}"] = 100 * (d["TA"] > d["FA"]).mean()
        result[f"Neg. {label}"] = 100 * (d["TN"] > d["FN"]).mean()
    return result

table14 = pd.DataFrame([
    {"Model": "BERT Base",  **nat_table14(nat_base)},
    {"Model": "BERT Large", **nat_table14(nat_large)},
])
table14 = table14[["Model", "Aff. NT", "Neg. NT", "Aff. LN", "Neg. LN"]]
display(table14)

print("Paper target:")
print("BERT Base : 62.5, 87.5, 75.0, 0.0")
print("BERT Large: 75.0, 100.0, 75.0, 0.0")

,Model,Aff. NT,Neg. NT,Aff. LN,Neg. LN
0,BERT Base,62.5,87.5,75.0,0.0
1,BERT Large,75.0,100.0,75.0,0.0


Paper target:
BERT Base : 62.5, 87.5, 75.0, 0.0
BERT Large: 75.0, 100.0, 75.0, 0.0


In [ ]:
def show_context_predictions(scored, contexts):
    rows = []
    for context in contexts:
        match = scored[scored.context == context]
        if len(match):
            r = match.iloc[0]
            rows.append({
                "Context": context.replace("[MASK]", "____"),
                "BERT Large predictions": ", ".join(r.top_tokens[:5])
            })
    return pd.DataFrame(rows)

table13_contexts = [
    "A robin is a [MASK]",
    "A daisy is a [MASK]",
    "A hammer is a [MASK]",
    "A hammer is an [MASK]",
    "A robin is not a [MASK]",
    "A daisy is not a [MASK]",
    "A hammer is not a [MASK]",
    "A hammer is not an [MASK]",
]

table13 = show_context_predictions(simp_large, table13_contexts)
display(table13)

,Context,BERT Large predictions
0,A robin is a ____,"bird, robin, person, hunter, pigeon"
1,A daisy is a ____,"daisy, rose, flower, berry, tree"
2,A hammer is a ____,"hammer, tool, weapon, nail, device"
3,A hammer is an ____,"object, instrument, axe, implement, explosive"
4,A robin is not a ____,"robin, bird, penguin, man, fly"
5,A daisy is not a ____,"daisy, rose, flower, lily, cherry"
6,A hammer is not a ____,"hammer, weapon, tool, gun, rock"
7,A hammer is not an ____,"object, instrument, axe, animal, artifact"


In [ ]:
table15_contexts = [
    "Most smokers find that quitting is very [MASK]",
    "Most smokers find that quitting isn't very [MASK]",
    "A fast food dinner on a first date is very [MASK]",
    "A fast food dinner on a first date isn't very [MASK]",
]

table15 = show_context_predictions(nat_large, table15_contexts)
display(table15)

,Context,BERT Large predictions
0,Most smokers find that quitting is very ____,"difficult, easy, effective, dangerous, hard"
1,Most smokers find that quitting isn't very ____,"effective, easy, attractive, difficult, succes..."
2,A fast food dinner on a first date is very ____,"good, nice, common, romantic, attractive"
3,A fast food dinner on a first date isn't very ...,"nice, good, romantic, appealing, exciting"


In [ ]:
def audit(scored):
    w = wide_probs(scored)
    w["TA>FA"] = w["TA"] > w["FA"]
    w["TN>FN"] = w["TN"] > w["FN"]
    w["TA-FA"] = w["TA"] - w["FA"]
    w["TN-FN"] = w["TN"] - w["FN"]
    return w

print("SIMP — BERT Base")
display(audit(simp_base))

print("SIMP — BERT Large")
display(audit(simp_large))

print("NAT — BERT Base")
display(audit(nat_base))

print("NAT — BERT Large")
display(audit(nat_large))

SIMP — BERT Base


cond,item,FA,FN,TA,TN,TA>FA,TN>FN,TA-FA,TN-FN
0,0,0.000454,0.282158,0.458702,7.007256e-05,True,False,0.458248,-0.282088
1,1,0.000294,0.372949,0.468620,7.392465e-05,True,False,0.468326,-0.372875
2,10,0.000044,0.007566,0.001668,6.144525e-04,True,False,0.001624,-0.006952
3,11,0.000435,0.012711,0.031981,3.654529e-04,True,False,0.031546,-0.012346
4,12,0.000578,0.177014,0.325408,4.986854e-04,True,False,0.324830,-0.176515
5,13,0.000004,0.022030,0.159117,4.140205e-06,True,False,0.159113,-0.022025
6,14,0.000095,0.238066,0.243470,1.194748e-04,True,False,0.243375,-0.237947
7,15,0.000016,0.103839,0.239107,4.882722e-06,True,False,0.239090,-0.103835
8,16,0.000005,0.002389,0.018337,3.913319e-07,True,False,0.018333,-0.002389
9,17,0.000120,0.013464,0.055227,1.260634e-05,True,False,0.055107,-0.013451


SIMP — BERT Large


cond,item,FA,FN,TA,TN,TA>FA,TN>FN,TA-FA,TN-FN
0,0,0.000055,0.539623,0.825476,0.000059,True,False,0.825421,-0.539563
1,1,0.000037,0.350676,0.597984,0.000058,True,False,0.597947,-0.350618
2,10,0.004687,0.015102,0.040064,0.003551,True,False,0.035377,-0.011551
3,11,0.000148,0.002650,0.041610,0.000089,True,False,0.041462,-0.002561
4,12,0.001689,0.095098,0.151806,0.002493,True,False,0.150117,-0.092605
5,13,0.000065,0.052297,0.500279,0.000082,True,False,0.500214,-0.052215
6,14,0.000148,0.066921,0.167242,0.000285,True,False,0.167094,-0.066637
7,15,0.000003,0.036160,0.029305,0.000007,True,False,0.029302,-0.036153
8,16,0.000004,0.008605,0.031087,0.000008,True,False,0.031083,-0.008597
9,17,0.000003,0.080459,0.217257,0.000033,True,False,0.217254,-0.080427


NAT — BERT Base


cond,item,FA,FN,TA,TN,TA>FA,TN>FN,TA-FA,TN-FN
0,0,0.038337,0.235215,0.109465,0.061657,True,False,0.071128,-0.173558
1,1,0.001107,0.014538,0.024116,0.039021,True,True,0.023009,0.024483
2,10,0.000092,0.159419,0.036679,0.010483,True,False,0.036587,-0.148936
3,11,0.000594,0.062366,0.002340,0.006485,True,False,0.001747,-0.055881
4,12,0.001750,0.009430,0.007406,0.000855,True,False,0.005655,-0.008575
5,13,0.000499,0.002088,0.000259,0.000066,False,False,-0.000240,-0.002023
6,14,0.002797,0.175657,0.245660,0.002639,True,False,0.242863,-0.173018
7,15,0.001028,0.015686,0.025153,0.000950,True,False,0.024126,-0.014736
8,2,0.000123,0.012109,0.000409,0.027839,True,True,0.000286,0.015730
9,3,0.023134,0.003575,0.004445,0.177035,False,True,-0.018689,0.173460


NAT — BERT Large


cond,item,FA,FN,TA,TN,TA>FA,TN>FN,TA-FA,TN-FN
0,0,0.150971,0.147417,0.486117,0.384907,True,True,0.335145,0.237490
1,1,0.002638,0.019196,0.180071,0.060117,True,True,0.177433,0.040920
2,10,0.000127,0.221379,0.006098,0.001321,True,False,0.005971,-0.220057
3,11,0.000814,0.010763,0.004963,0.001637,True,False,0.004149,-0.009126
4,12,0.005743,0.090414,0.012966,0.000201,True,False,0.007224,-0.090212
5,13,0.000338,0.005825,0.000118,0.000141,False,False,-0.000221,-0.005684
6,14,0.001726,0.202487,0.532831,0.002655,True,False,0.531106,-0.199832
7,15,0.000551,0.095063,0.015363,0.002058,True,False,0.014812,-0.093004
8,2,0.000110,0.023491,0.001340,0.267007,True,True,0.001230,0.243516
9,3,0.001160,0.002768,0.008258,0.213558,True,True,0.007098,0.210790


In [ ]:
# ---- Upload NEG-136-SIMP-BAL.tsv (Colab file picker) ----
BAL_FILENAME = "NEG-136-SIMP-BAL-RARE.tsv"
BAL_PATH = f"/content/{BAL_FILENAME}"

if not os.path.exists(BAL_PATH):          # skip the picker if it's already uploaded
    from google.colab import files
    uploaded = files.upload()             # choose NEG-136-SIMP-BAL.tsv
    BAL_PATH = f"/content/{next(iter(uploaded))}"

bal_raw = pd.read_csv(BAL_PATH, sep="\t")
print("BAL path:", BAL_PATH)
print("BAL:", bal_raw.shape)              # expected (70, 5)
assert list(bal_raw.columns) == list(simp_raw.columns), bal_raw.columns
display(bal_raw.head())

# Balance check: each category should appear equally often as target_aff and as target_neg
display(pd.DataFrame({"as target_aff": bal_raw.target_aff.value_counts(),
                      "as target_neg": bal_raw.target_neg.value_counts()}))

Saving NEG-136-SIMP-BAL-RARE.tsv to NEG-136-SIMP-BAL-RARE.tsv
BAL path: /content/NEG-136-SIMP-BAL-RARE.tsv
BAL: (70, 5)


,item,context_aff,context_neg,target_aff,target_neg
0,1,A granary is (a|an),A granary is not (a|an),building,fruit
1,2,A banana is (a|an),A banana is not (a|an),fruit,flower
2,3,A hotel is (a|an),A hotel is not (a|an),building,fish
3,4,A moth is (a|an),A moth is not (a|an),insect,bird
4,5,A chisel is (a|an),A chisel is not (a|an),tool,flower


,as target_aff,as target_neg
bird,7,7
building,7,7
fish,7,7
flower,7,7
fruit,7,7
insect,7,7
tool,7,7
tree,7,7
vegetable,7,7
vehicle,7,7


In [ ]:

bal = process_negsimp(BAL_PATH)

assert len(bal) == len(bal_raw) * 4
assert set(bal.cond) == {"TA","FN","FA","TN"}

for name, tok in [("base", tok_base), ("large", tok_large)]:
    bad = [t for t in sorted(bal.target.unique()) if tok.tokenize(t) != [t]]
    print(f"{name}: targets that are NOT single tokens -> {bad if bad else 'none'}")

display(bal.head(8))

base: targets that are NOT single tokens -> none
large: targets that are NOT single tokens -> none


,item,cond,context,target,expected
0,1,TA,A granary is a [MASK],building,building
1,1,FN,A granary is not a [MASK],building,fruit
2,1,FA,A granary is a [MASK],fruit,building
3,1,TN,A granary is not a [MASK],fruit,fruit
4,2,TA,A banana is a [MASK],fruit,fruit
5,2,FN,A banana is not a [MASK],fruit,flower
6,2,FA,A banana is a [MASK],flower,fruit
7,2,TN,A banana is not a [MASK],flower,flower


In [ ]:

example = bal.query("context == 'A robin is a [MASK]' and cond == 'TA'").iloc[0]
_, mi, toks = prep_input(example.context, tok_large)
print(example.context)
print(toks)
print("mask index:", mi)

A robin is a [MASK]
['[CLS]', 'a', 'robin', 'is', 'a', '[MASK]', '.', '[SEP]']
mask index: 5


In [ ]:
bal_base  = score_dataframe(bal, bert_base,  tok_base,  topk=5)
bal_large = score_dataframe(bal, bert_large, tok_large, topk=5)
print("Scoring complete (BAL).")

Scoring complete (BAL).


In [ ]:

table11_bal = pd.DataFrame([
    ["BERT Base",  *simp_accuracy(bal_base, 1),  *simp_accuracy(bal_base, 5)],
    ["BERT Large", *simp_accuracy(bal_large, 1), *simp_accuracy(bal_large, 5)],
], columns=["Model", "k1_pct", "k1_correct", "k1_n", "k5_pct", "k5_correct", "k5_n"])

print("Table 11 — NEG-136-SIMP-BAL")
display(table11_bal)

Table 11 — NEG-136-SIMP-BAL


,Model,k1_pct,k1_correct,k1_n,k5_pct,k5_correct,k5_n
0,BERT Base,35.714286,25,70,88.571429,62,70
1,BERT Large,55.714286,39,70,97.142857,68,70


In [ ]:

table12_bal = pd.DataFrame([
    {"Model": "BERT Base",  **neg_sensitivity(bal_base)},
    {"Model": "BERT Large", **neg_sensitivity(bal_large)},
])
print("Table 12 — NEG-136-SIMP-BAL")
display(table12_bal)

table12_bal_threshold = pd.DataFrame([
    {"Model": "BERT Base",  **neg_sensitivity(bal_base,  threshold=.01)},
    {"Model": "BERT Large", **neg_sensitivity(bal_large, threshold=.01)},
])
print("Table 12 (threshold 0.01) — NEG-136-SIMP-BAL")
display(table12_bal_threshold)

Table 12 — NEG-136-SIMP-BAL


,Model,Affirmative,Negative,Aff_correct,Neg_correct,n
0,BERT Base,100.000000,1.428571,70,1,70
1,BERT Large,98.571429,1.428571,69,1,70


Table 12 (threshold 0.01) — NEG-136-SIMP-BAL


,Model,Affirmative,Negative,Aff_correct,Neg_correct,n
0,BERT Base,92.857143,1.428571,65,1,70
1,BERT Large,95.714286,1.428571,67,1,70


In [ ]:

table13_bal = show_context_predictions(bal_large, table13_contexts)
print("Table 13 — NEG-136-SIMP-BAL (BERT Large)")
display(table13_bal)

Table 13 — NEG-136-SIMP-BAL (BERT Large)


,Context,BERT Large predictions
0,A robin is a ____,"bird, robin, person, hunter, pigeon"
1,A daisy is a ____,"daisy, rose, flower, berry, tree"
2,A hammer is a ____,"hammer, tool, weapon, nail, device"
3,A hammer is an ____,"object, instrument, axe, implement, explosive"
4,A robin is not a ____,"robin, bird, penguin, man, fly"
5,A daisy is not a ____,"daisy, rose, flower, lily, cherry"
6,A hammer is not a ____,"hammer, weapon, tool, gun, rock"
7,A hammer is not an ____,"object, instrument, axe, animal, artifact"


In [ ]:

!pip -q install wordfreq
from wordfreq import zipf_frequency

ETTINGER = {"trout", "salmon", "ant", "bee", "robin", "sparrow", "oak", "pine", "rose",
            "daisy", "carrot", "pea", "hammer", "saw", "car", "truck", "hotel", "house"}

def table13_full(scored, raw, model_name):
    ta = scored[scored.cond == "TA"].set_index("item")
    fn = scored[scored.cond == "FN"].set_index("item")
    rows = []
    for _, r in raw.iterrows():
        item = str(r["item"])
        hyponym = re.match(r"^(?:A|An)\s+(.+?)\s+is\b", r["context_aff"], flags=re.I).group(1).lower()
        cat = r["target_aff"]
        top_aff = ta.loc[item, "top_tokens"][:5]
        top_neg = fn.loc[item, "top_tokens"][:5]
        z = round(zipf_frequency(hyponym, "en"), 2)
        rows.append({
            "Model": model_name,
            "item": int(item),
            "hyponym": hyponym,
            "category": cat,
            "zipf": z,
            "frequency": "rare (<3)" if z < 3 else "not rare",
            "Ettinger": hyponym in ETTINGER,
            "Affirmative context": ta.loc[item, "context"].replace("[MASK]", "____"),
            "Top-5 (affirmative)": ", ".join(top_aff),
            "category in top-5": cat in top_aff,
            "category is top-1": top_aff[0] == cat,
            "Negated context": fn.loc[item, "context"].replace("[MASK]", "____"),
            "Top-5 (negated)": ", ".join(top_neg),
            "category STILL in top-5 after 'not'": cat in top_neg,
        })
    return pd.DataFrame(rows).sort_values("item").reset_index(drop=True)

t13_base  = table13_full(bal_base,  bal_raw, "BERT Base")
t13_large = table13_full(bal_large, bal_raw, "BERT Large")
t13_all   = pd.concat([t13_base, t13_large], ignore_index=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
for name, t in [("BERT Large", t13_large), ("BERT Base", t13_base)]:
    print(f"\nTable 13 — NEG-136-SIMP-BAL-RARE, {name}, all items (n={len(t)})")
    display(t.drop(columns="Model"))


def summarise(t, by=None):
    g = t.groupby(by) if by else t.groupby(lambda _: "all items")
    return g.agg(n=("item", "size"),
                 aff_top5=("category in top-5", "sum"),
                 aff_top1=("category is top-1", "sum"),
                 still_top5_after_not=("category STILL in top-5 after 'not'", "sum"))

for name, t in [("BERT Large", t13_large), ("BERT Base", t13_base)]:
    print(f"\n{name} — overall");            display(summarise(t))
    print(f"{name} — rare vs not rare");      display(summarise(t, "frequency"))
    print(f"{name} — per category");          display(summarise(t, "category"))


fails = t13_all[~t13_all["category in top-5"]]
print("\nItems where the category is NOT in the affirmative top-5:")
display(fails[["Model", "item", "hyponym", "category", "zipf", "frequency", "Top-5 (affirmative)"]])


t13_all.to_csv("table13_full_NEG-136-SIMP-BAL-RARE.csv", index=False)
try:
    from google.colab import files
    files.download("table13_full_NEG-136-SIMP-BAL-RARE.csv")
except ImportError:
    print("Saved table13_full_NEG-136-SIMP-BAL-RARE.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 19.7 MB/s eta 0:00:00

Table 13 — NEG-136-SIMP-BAL-RARE, BERT Large, all items (n=70)


,item,hyponym,category,zipf,frequency,Ettinger,Affirmative context,Top-5 (affirmative),category in top-5,category is top-1,Negated context,Top-5 (negated),category STILL in top-5 after 'not'
0,1,granary,building,2.57,rare (<3),False,A granary is a ____,"building, warehouse, barn, house, structure",True,True,A granary is not a ____,"building, house, hotel, farm, barn",True
1,2,banana,fruit,4.00,not rare,False,A banana is a ____,"banana, tree, fruit, vegetable, plant",True,False,A banana is not a ____,"banana, vegetable, tree, fruit, peach",True
2,3,hotel,building,4.95,not rare,True,A hotel is a ____,"hotel, resort, building, house, motel",True,False,A hotel is not a ____,"hotel, motel, resort, casino, restaurant",False
3,4,moth,insect,3.50,not rare,False,A moth is an ____,"insect, animal, object, ant, organism",True,True,A moth is not an ____,"animal, insect, object, ant, egg",True
4,5,chisel,tool,2.96,rare (<3),False,A chisel is a ____,"tool, hammer, knife, blade, weapon",True,True,A chisel is not a ____,"tool, knife, hammer, blade, weapon",True
5,6,train,vehicle,4.96,not rare,False,A train is a ____,"train, vehicle, locomotive, car, railroad",True,False,A train is not a ____,"train, car, boat, bus, ship",False
6,7,church,building,5.16,not rare,False,A church is a ____,"church, cathedral, parish, monastery, temple",False,False,A church is not a ____,"church, temple, cathedral, building, monastery",True
7,8,cabbage,vegetable,3.55,not rare,False,A cabbage is a ____,"cabbage, vegetable, carrot, plant, potato",True,False,A cabbage is not a ____,"cabbage, vegetable, carrot, bean, dog",True
8,9,kestrel,bird,2.47,rare (<3),False,A kestrel is a ____,"bird, raven, pigeon, crow, hawk",True,True,A kestrel is not a ____,"bird, raven, butterfly, crow, pigeon",True
9,10,school,building,5.71,not rare,False,A school is a ____,"school, college, building, community, structure",True,False,A school is not a ____,"school, church, university, home, place",False



Table 13 — NEG-136-SIMP-BAL-RARE, BERT Base, all items (n=70)


,item,hyponym,category,zipf,frequency,Ettinger,Affirmative context,Top-5 (affirmative),category in top-5,category is top-1,Negated context,Top-5 (negated),category STILL in top-5 after 'not'
0,1,granary,building,2.57,rare (<3),False,A granary is a ____,"building, container, structure, refrigerator, compound",True,True,A granary is not a ____,"building, kitchen, structure, refrigerator, container",True
1,2,banana,fruit,4.00,not rare,False,A banana is a ____,"banana, fruit, plant, tree, vegetable",True,False,A banana is not a ____,"banana, fruit, tree, vegetable, bananas",True
2,3,hotel,building,4.95,not rare,True,A hotel is a ____,"hotel, business, resort, restaurant, building",True,False,A hotel is not a ____,"hotel, resort, motel, restaurant, business",False
3,4,moth,insect,3.50,not rare,False,A moth is an ____,"insect, animal, illusion, asteroid, ant",True,True,A moth is not an ____,"animal, insect, exception, organism, entity",True
4,5,chisel,tool,2.96,rare (<3),False,A chisel is a ____,"tool, weapon, hammer, blade, knife",True,True,A chisel is not a ____,"tool, hammer, weapon, knife, stone",True
5,6,train,vehicle,4.96,not rare,False,A train is a ____,"train, locomotive, car, thing, machine",False,False,A train is not a ____,"train, locomotive, car, vehicle, passenger",True
6,7,church,building,5.16,not rare,False,A church is a ____,"village, church, parish, cemetery, commune",False,False,A church is not a ____,"church, parish, community, sanctuary, religion",False
7,8,cabbage,vegetable,3.55,not rare,False,A cabbage is a ____,"cabbage, vegetable, plant, potato, soup",True,False,A cabbage is not a ____,"cabbage, vegetable, potato, tomato, bean",True
8,9,kestrel,bird,2.47,rare (<3),False,A kestrel is a ____,"horse, dog, bird, mammal, human",True,False,A kestrel is not a ____,"dog, bird, hunter, horse, cat",True
9,10,school,building,5.71,not rare,False,A school is a ____,"school, community, university, village, college",False,False,A school is not a ____,"school, community, college, university, village",False



BERT Large — overall


...

,n,aff_top5,aff_top1,still_top5_after_not
category,,,,
bird,7,7,5,7
building,7,5,1,4
fish,7,5,2,6
flower,7,7,3,7
fruit,7,7,1,6
insect,7,6,3,6
tool,7,7,4,7
tree,7,7,5,7
vegetable,7,6,0,5



Items where the category is NOT in the affirmative top-5:


,Model,item,hyponym,category,zipf,frequency,Top-5 (affirmative)
5,BERT Base,6,train,vehicle,4.96,not rare,"train, locomotive, car, thing, machine"
6,BERT Base,7,church,building,5.16,not rare,"village, church, parish, cemetery, commune"
9,BERT Base,10,school,building,5.71,not rare,"school, community, university, village, college"
29,BERT Base,30,cauliflower,vegetable,3.20,not rare,"plant, flower, ##flower, shrub, perennial"
31,BERT Base,32,halibut,fish,2.71,rare (<3),"cocktail, salad, cake, pastry, dog"
34,BERT Base,35,moped,vehicle,2.78,rare (<3),"bicycle, motorcycle, horse, dog, bike"
51,BERT Base,52,termite,insect,2.78,rare (<3),"asteroid, explosive, object, element, aggregate"
56,BERT Base,57,haddock,fish,2.83,rare (<3),"horse, dog, noun, ship, name"
76,BERT Large,7,church,building,5.16,not rare,"church, cathedral, parish, monastery, temple"
138,BERT Large,69,peach,fruit,3.77,not rare,"peach, plum, berry, lemon, rose"


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

print("SIMP-BAL — BERT Base")
display(audit(bal_base))

print("SIMP-BAL — BERT Large")
display(audit(bal_large))

SIMP-BAL — BERT Base


cond,item,FA,FN,TA,TN,TA>FA,TN>FN,TA-FA,TN-FN
0,1,2.020712e-05,0.133582,0.155367,6.996226e-06,True,False,0.155347,-0.133575
1,10,3.585854e-04,0.001903,0.013419,7.779888e-05,True,False,0.013061,-0.001825
2,11,1.967524e-04,0.072219,0.054946,6.657573e-04,True,False,0.054749,-0.071553
3,12,4.208911e-05,0.927698,0.797508,4.915193e-05,True,False,0.797466,-0.927649
4,13,7.137485e-05,0.005584,0.057710,8.337881e-06,True,False,0.057639,-0.005575
5,14,1.713753e-04,0.063732,0.032233,4.419597e-04,True,False,0.032061,-0.063290
6,15,3.233316e-04,0.698534,0.836194,1.739339e-04,True,False,0.835871,-0.698360
7,16,2.939885e-04,0.372949,0.468620,7.392465e-05,True,False,0.468326,-0.372875
8,17,1.316093e-04,0.001202,0.014942,1.407258e-06,True,False,0.014810,-0.001200
9,18,9.493683e-05,0.542941,0.770262,2.896631e-04,True,False,0.770167,-0.542652


SIMP-BAL — BERT Large


cond,item,FA,FN,TA,TN,TA>FA,TN>FN,TA-FA,TN-FN
0,1,2.075413e-05,0.051361,0.247394,0.000010,True,False,0.247373,-0.051351
1,10,3.383576e-05,0.010343,0.028303,0.000189,True,False,0.028269,-0.010153
2,11,1.615921e-03,0.011399,0.086995,0.000451,True,False,0.085379,-0.010948
3,12,1.830772e-04,0.774649,0.744313,0.000085,True,False,0.744129,-0.774564
4,13,1.557600e-04,0.013861,0.169594,0.000260,True,False,0.169438,-0.013601
5,14,1.991293e-05,0.099095,0.431361,0.000078,True,False,0.431341,-0.099017
6,15,8.261104e-04,0.090728,0.495718,0.000145,True,False,0.494892,-0.090583
7,16,3.734195e-05,0.350676,0.597984,0.000058,True,False,0.597947,-0.350618
8,17,4.836871e-05,0.005710,0.023273,0.000079,True,False,0.023224,-0.005631
9,18,4.276712e-04,0.209651,0.522395,0.000156,True,False,0.521968,-0.209495


In [ ]:

cmp11 = pd.concat([table11.assign(Dataset="NEG-136-SIMP"),
                   table11_bal.assign(Dataset="NEG-136-SIMP-BAL")])
cmp11 = cmp11[["Dataset", "Model", "k1_pct", "k1_correct", "k1_n", "k5_pct", "k5_correct", "k5_n"]]
print("Table 11 comparison")
display(cmp11.reset_index(drop=True))

cmp12 = pd.concat([table12.assign(Dataset="NEG-136-SIMP", Threshold="none"),
                   table12_threshold.assign(Dataset="NEG-136-SIMP", Threshold="0.01"),
                   table12_bal.assign(Dataset="NEG-136-SIMP-BAL", Threshold="none"),
                   table12_bal_threshold.assign(Dataset="NEG-136-SIMP-BAL", Threshold="0.01")])
cmp12 = cmp12[["Dataset", "Threshold", "Model", "Affirmative", "Negative", "Aff_correct", "Neg_correct", "n"]]
print("Table 12 comparison")
display(cmp12.reset_index(drop=True))

Table 11 comparison


,Dataset,Model,k1_pct,k1_correct,k1_n,k5_pct,k5_correct,k5_n
0,NEG-136-SIMP,BERT Base,38.888889,7,18,100.000000,18,18
1,NEG-136-SIMP,BERT Large,44.444444,8,18,100.000000,18,18
2,NEG-136-SIMP-BAL,BERT Base,35.714286,25,70,88.571429,62,70
3,NEG-136-SIMP-BAL,BERT Large,55.714286,39,70,97.142857,68,70


Table 12 comparison


,Dataset,Threshold,Model,Affirmative,Negative,Aff_correct,Neg_correct,n
0,NEG-136-SIMP,none,BERT Base,100.000000,0.000000,18,0,18
1,NEG-136-SIMP,none,BERT Large,100.000000,0.000000,18,0,18
2,NEG-136-SIMP,0.01,BERT Base,94.444444,0.000000,17,0,18
3,NEG-136-SIMP,0.01,BERT Large,100.000000,0.000000,18,0,18
4,NEG-136-SIMP-BAL,none,BERT Base,100.000000,1.428571,70,1,70
5,NEG-136-SIMP-BAL,none,BERT Large,98.571429,1.428571,69,1,70
6,NEG-136-SIMP-BAL,0.01,BERT Base,92.857143,1.428571,65,1,70
7,NEG-136-SIMP-BAL,0.01,BERT Large,95.714286,1.428571,67,1,70


In [ ]:
# ---- Optional extra (not in the paper): results per category, which the balanced design makes comparable ----
def per_category(scored, raw, model_name):
    w = wide_probs(scored)
    w["category"] = w["item"].map(dict(zip(raw["item"].astype(str), raw["target_aff"])))
    ta = scored[scored.cond == "TA"].copy()
    ta["k1"] = [row.expected in row.top_tokens[:1] for _, row in ta.iterrows()]
    ta["k5"] = [row.expected in row.top_tokens[:5] for _, row in ta.iterrows()]
    w = w.merge(ta[["item", "k1", "k5"]], on="item")
    g = w.groupby("category")[["TA", "FA", "TN", "FN", "k1", "k5"]].apply(lambda d: pd.Series({
        "n": len(d),
        "k1_pct": 100 * d.k1.mean(),
        "k5_pct": 100 * d.k5.mean(),
        "Affirmative": 100 * (d.TA > d.FA).mean(),
        "Negative": 100 * (d.TN > d.FN).mean(),
    }))
    return g.reset_index().assign(Model=model_name)

per_cat_bal = pd.concat([per_category(bal_base, bal_raw, "BERT Base"),
                         per_category(bal_large, bal_raw, "BERT Large")])
display(per_cat_bal[["Model", "category", "n", "k1_pct", "k5_pct", "Affirmative", "Negative"]].reset_index(drop=True))

,Model,category,n,k1_pct,k5_pct,Affirmative,Negative
0,BERT Base,bird,7.0,71.428571,100.000000,100.000000,0.000000
1,BERT Base,building,7.0,14.285714,71.428571,100.000000,0.000000
2,BERT Base,fish,7.0,28.571429,71.428571,100.000000,0.000000
3,BERT Base,flower,7.0,42.857143,100.000000,100.000000,0.000000
4,BERT Base,fruit,7.0,14.285714,100.000000,100.000000,14.285714
5,BERT Base,insect,7.0,42.857143,85.714286,100.000000,0.000000
6,BERT Base,tool,7.0,57.142857,100.000000,100.000000,0.000000
7,BERT Base,tree,7.0,71.428571,100.000000,100.000000,0.000000
8,BERT Base,vegetable,7.0,0.000000,85.714286,100.000000,0.000000
9,BERT Base,vehicle,7.0,14.285714,71.428571,100.000000,0.000000
